<a href="https://colab.research.google.com/github/ProdiptoPantho/BracU_UniNesT/blob/main/lightnovel_crawler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LightNovel Crawler (Custom UI)

In [2]:
# @markdown <font size =5 color="yellow"> ⬅️ Click Here to install</font>
from google.colab import output
!curl -LsSf https://astral.sh/uv/install.sh | sh
output.clear()
!uv pip install -U lightnovel-crawler
# !uv pip install gofilepy-api

output.clear()
print("Done!")

Done!


In [3]:
# @markdown <font size =5 color="yellow"> ⬅️ Click Here to start the server</font>
import subprocess, threading, time, socket

PORT = 7860

proc = subprocess.Popen(
    ["python", "-m", "lncrawl", "-ll", "server", "--host", "0.0.0.0", "--port", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

def consume_logs():
    for _ in proc.stdout:
        pass
threading.Thread(target=consume_logs, daemon=True).start()

def is_port_open(port, host="127.0.0.1"):
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

# Wait until port is actually open (max 60s)
steps = ["⠋","⠙","⠹","⠸","⠼","⠴","⠦","⠧","⠇","⠏"]
i = 0
for _ in range(120):
    print(f"\r{steps[i % len(steps)]} Starting lncrawl server...", end="", flush=True)
    if is_port_open(PORT):
        break
    time.sleep(0.5)
    i += 1
else:
    print("\r❌ Server failed to start. Check if port is in use.")
    proc.kill()
    raise SystemExit

from google.colab.output import eval_js
url = eval_js(f"google.colab.kernel.proxyPort({PORT})")

print("\r" + " " * 40)
print("━" * 50)
print("  📚  LightNovel Crawler — Server Ready")
print("━" * 50)
print(f"  🌐  URL   : {url}")
print(f"  👤  User  : admin")
print(f"  🔑  Pass  : admin")
print("━" * 50)
print("  ⚠️  Session ends when Colab disconnects. KEEP THIS TAB OPEN UNTIL YOU FINISH!!")
print("━" * 50)

                                        
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📚  LightNovel Crawler — Server Ready
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🌐  URL   : https://7860-m-s-kkb-use1c0-3q672b2lpyrfz-c.us-east1-0.prod.colab.dev
  👤  User  : admin
  🔑  Pass  : admin
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ⚠️  Session ends when Colab disconnects. KEEP THIS TAB OPEN UNTIL YOU FINISH!!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## List of Working Sources sorted by speed.
⭐ = Personal Recommedation  

<font color="red" size="4"> Note:</font> <font size="3">working doesn't mean that it has the novel you want or the novel isn't missing chapters in that source, so always take notice of the the number of chapters while downloading.

In [3]:
#@markdown ## <font color="#f9e2af" size=5>⬅️ Fetch Working Sources</font>

import pandas as pd
from urllib.parse import urlparse, parse_qs
from IPython.display import display, HTML
import re

# ──────────────────────────────────────────────
#  Google Sheets Fetcher
# ──────────────────────────────────────────────

def fetch_google_sheet(sheet_url: str) -> pd.DataFrame:
    """Fetch a Google Sheets document as a DataFrame via CSV export."""
    try:
        parsed = urlparse(sheet_url)
        path_parts = parsed.path.split("/")

        if len(path_parts) < 4:
            raise ValueError("Invalid Google Sheets URL — cannot extract document ID.")

        doc_id = path_parts[3]

        # Resolve gid from query string or fragment
        gid = (
            parse_qs(parsed.query).get("gid")
            or parse_qs(parsed.fragment).get("gid")
            or ["0"]
        )[0]

        export_url = (
            f"https://docs.google.com/spreadsheets/d/{doc_id}"
            f"/export?format=csv&gid={gid}"
        )

        return pd.read_csv(export_url)

    except Exception as exc:
        print(f"\033[91m✖ Error fetching Google Sheet:\033[0m {exc}")
        return pd.DataFrame()


# ──────────────────────────────────────────────
#  Formatting Helpers
# ──────────────────────────────────────────────

_URL_RE = re.compile(r"^https?://\S+$")


def _format_cell(value) -> str:
    """Return an HTML-safe string for a single cell value."""
    if pd.isna(value):
        return ""

    text = str(value)

    # Clickable link
    if _URL_RE.match(text):
        short = text if len(text) <= 60 else text[:57] + "…"
        return f'<a href="{text}" target="_blank" style="color:#89dceb;text-decoration:none;">{short}</a>'

    # Truncate long text
    if len(text) > 50:
        return text[:47] + "…"

    return text


def _format_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Convert numeric columns to clean display strings in-place."""
    df = df.copy()
    for col in df.select_dtypes(include="number").columns:
        is_int_like = df[col].dropna().apply(float.is_integer).all()
        if is_int_like:
            df[col] = df[col].apply(lambda v: "" if pd.isna(v) else str(int(v)))
        else:
            df[col] = df[col].apply(lambda v: "" if pd.isna(v) else f"{v:.2f}")
    return df


# ──────────────────────────────────────────────
#  HTML Table Renderer (Catppuccin Mocha theme)
# ──────────────────────────────────────────────

_TABLE_CSS = """
<style>
  .src-table-wrap {
    overflow-x: auto;
    margin: 16px 0;
  }
  .src-table {
    border-collapse: collapse;
    width: max-content;
    min-width: 60%;
    font-family: 'Inter', 'Segoe UI', system-ui, sans-serif;
    font-size: 14px;
    border: 1px solid #45475a;
    border-radius: 10px;
    overflow: hidden;
  }
  .src-table thead th {
    background: linear-gradient(135deg, #89b4fa 0%, #74c7ec 100%);
    color: #1e1e2e;
    font-weight: 700;
    font-size: 15px;
    padding: 12px 16px;
    white-space: nowrap;
    text-transform: uppercase;
    letter-spacing: 0.4px;
    border-bottom: 2px solid #585b70;
  }
  .src-table tbody td {
    background: #1e1e2e;
    color: #cdd6f4;
    padding: 10px 16px;
    border-bottom: 1px solid #313244;
    white-space: nowrap;
  }
  .src-table tbody tr:nth-child(even) td {
    background: #181825;
  }
  .src-table tbody tr:hover td {
    background: #313244;
  }
  .src-footer {
    color: #a6adc8;
    font-size: 13px;
    margin-top: 6px;
    font-style: italic;
  }
</style>
"""


def display_table(df: pd.DataFrame, max_rows: int = 500) -> None:
    """Render a DataFrame as a themed HTML table inside the notebook."""
    if df.empty:
        display(HTML('<p style="color:#f38ba8;font-weight:600;">⚠ No data available.</p>'))
        return

    show_df = df.head(max_rows)
    truncated = len(df) > max_rows

    # ---- Build HTML with list join (faster than repeated concatenation) ----
    parts: list[str] = [_TABLE_CSS, '<div class="src-table-wrap">', '<table class="src-table">']

    # Header
    parts.append("<thead><tr>")
    parts.extend(f"<th>{col}</th>" for col in show_df.columns)
    parts.append("</tr></thead>")

    # Body — iterate underlying NumPy matrix for speed
    parts.append("<tbody>")
    values = show_df.values
    for row in values:
        parts.append("<tr>")
        parts.extend(f"<td>{_format_cell(v)}</td>" for v in row)
        parts.append("</tr>")
    parts.append("</tbody></table></div>")

    if truncated:
        parts.append(f'<p class="src-footer">Showing {max_rows} of {len(df)} rows</p>')

    display(HTML("\n".join(parts)))


# ──────────────────────────────────────────────
#  Main
# ──────────────────────────────────────────────

def main():
    SHEET_URL = (
        "https://docs.google.com/spreadsheets/d/"
        "1SGfpvLFq7DzssklY90cmTKqQEBNeOBcAj1zSHDv-gfA/edit?usp=sharing"
    )

    df = fetch_google_sheet(SHEET_URL)
    df = _format_numeric_columns(df)
    display_table(df, max_rows=500)


if __name__ == "__main__":
    main()

"Sources (updated March 10, 2025)",Average Speed (CH/s),Notes
https://readnovelfull.com/,35.00,
https://www.wuxiaworld.com/,30.00,Needs subscription for locked novels
https://allnovel.org/,20.00,
https://raeitranslations.com/,19.00,
http://novelfull.com/,18.00,
https://wuxia.city/,18.00,
https://novelfull.net/,16.00,
https://novelhi.com/,16.00,
https://hostednovel.com/,15.00,
https://www.novels.pl/,13.00,
